In [1]:
# ============================================================================
# CELL 1: IMPORTS AND SETUP
# ============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import os
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


 Using device: cpu


In [2]:
# ============================================================================
# CELL 2: CONFIGURATION
# ============================================================================

class Config:
    """Training configuration"""
    
    # Paths
    DATA_DIR = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\gifgif-images-v1\gifgif-images"  # UPDATE THIS
    TRAIN_CSV = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\train_grouped.csv"          # UPDATE THIS
    VAL_CSV = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\val_grouped.csv"              # UPDATE THIS
    TEST_CSV = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\test_grouped.csv"            # UPDATE THIS
    MODEL_SAVE_DIR = "models/"
    
    # Model hyperparameters
    NUM_CLASSES = 6
    NUM_FRAMES = 3  # Number of frames to extract per GIF
    IMG_SIZE = 224
    
    # ConvGRU hidden dimensions
    HIDDEN_DIM_1 = 512  # First ConvGRU layer
    HIDDEN_DIM_2 = 256  # Second ConvGRU layer
    
    # Training hyperparameters
    BATCH_SIZE = 16      # Reduce if GPU memory is limited
    NUM_EPOCHS = 30
    LEARNING_RATE = 0.0001
    WEIGHT_DECAY = 1e-4
    
    # Early stopping
    PATIENCE = 7  # Stop if no improvement for 7 epochs
    
    # Data augmentation (training only)
    USE_AUGMENTATION = True
    
    # Emotion mapping
    EMOTION_TO_IDX = {
        'contempt': 0,
        'negative_intense': 1,
        'negative_subdued': 2,
        'positive_calm': 3,
        'positive_energetic': 4,
        'surprise': 5
    }
    IDX_TO_EMOTION = {v: k for k, v in EMOTION_TO_IDX.items()}

# Create model directory
os.makedirs(Config.MODEL_SAVE_DIR, exist_ok=True)

print(" Configuration:")
print(f"   Batch size: {Config.BATCH_SIZE}")
print(f"   Epochs: {Config.NUM_EPOCHS}")
print(f"   Learning rate: {Config.LEARNING_RATE}")
print(f"   Frames per GIF: {Config.NUM_FRAMES}")
print(f"   Emotion classes: {Config.NUM_CLASSES}")

print(Config.DATA_DIR)

 Configuration:
   Batch size: 16
   Epochs: 30
   Learning rate: 0.0001
   Frames per GIF: 3
   Emotion classes: 6
D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\gifgif-images-v1\gifgif-images


In [3]:
# ============================================================================
# CELL 3: CONVGRU CELL IMPLEMENTATION
# ============================================================================

class ConvGRUCell(nn.Module):
    """
    Convolutional GRU Cell for spatiotemporal feature modeling
    
    Based on: Ballas et al. "Delving Deeper into Convolutional Networks for 
              Learning Video Representations" (2015)
    """
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super(ConvGRUCell, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        padding = kernel_size // 2
        
        # Gates: reset, update, new
        self.conv_xr = nn.Conv2d(input_dim, hidden_dim, kernel_size, padding=padding)
        self.conv_hr = nn.Conv2d(hidden_dim, hidden_dim, kernel_size, padding=padding)
        
        self.conv_xz = nn.Conv2d(input_dim, hidden_dim, kernel_size, padding=padding)
        self.conv_hz = nn.Conv2d(hidden_dim, hidden_dim, kernel_size, padding=padding)
        
        self.conv_xn = nn.Conv2d(input_dim, hidden_dim, kernel_size, padding=padding)
        self.conv_hn = nn.Conv2d(hidden_dim, hidden_dim, kernel_size, padding=padding)
    
    def forward(self, x, h_prev):
        """
        Args:
            x: Input tensor [batch, input_dim, H, W]
            h_prev: Previous hidden state [batch, hidden_dim, H, W]
        Returns:
            h: New hidden state [batch, hidden_dim, H, W]
        """
        # Reset gate
        r = torch.sigmoid(self.conv_xr(x) + self.conv_hr(h_prev))
        
        # Update gate
        z = torch.sigmoid(self.conv_xz(x) + self.conv_hz(h_prev))
        
        # New gate (candidate)
        n = torch.tanh(self.conv_xn(x) + self.conv_hn(r * h_prev))
        
        # New hidden state
        h = (1 - z) * n + z * h_prev
        
        return h

print(" ConvGRUCell implementation complete")

 ConvGRUCell implementation complete


In [4]:
# ============================================================================
# CELL 4: RESNET-CONVGRU MODEL
# ============================================================================

class EmotionConvGRU(nn.Module):
    """
    ResNet50-ConvGRU model for GIF emotion recognition
    
    Architecture:
        1. ResNet50 feature extractor (per frame)
        2. 2-layer ConvGRU for temporal modeling
        3. Classification head
    
    Following: Zhang et al. 2022 architecture
    Adapted for: 6 emotion groups
    """
    def __init__(self, num_classes=6, num_frames=3, 
                 hidden_dim_1=512, hidden_dim_2=256,
                 dropout=0.5, pretrained=True):
        super(EmotionConvGRU, self).__init__()
        
        self.num_frames = num_frames
        self.hidden_dim_1 = hidden_dim_1
        self.hidden_dim_2 = hidden_dim_2
        
        # ResNet50 feature extractor (pretrained on ImageNet)
        resnet = models.resnet50(pretrained=pretrained)
        # Remove final avgpool and fc layers, keep spatial structure
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        # Output: [batch, 2048, 7, 7]
        
        # ConvGRU layers for temporal modeling
        self.convgru1 = ConvGRUCell(2048, hidden_dim_1, kernel_size=3)
        self.convgru2 = ConvGRUCell(hidden_dim_1, hidden_dim_2, kernel_size=3)
        
        # Global pooling
        self.pool = nn.AdaptiveAvgPool2d(1)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim_2, num_classes)
        )
        
        # Initialize weights
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize classifier weights"""
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        """
        Args:
            x: Input tensor [batch, num_frames, 3, 224, 224]
        Returns:
            output: Class logits [batch, num_classes]
        """
        batch_size = x.size(0)
        
        # Extract features from each frame
        frame_features = []
        for t in range(self.num_frames):
            feat = self.features(x[:, t])  # [batch, 2048, 7, 7]
            frame_features.append(feat)
        
        # Initialize hidden states
        h1 = torch.zeros(batch_size, self.hidden_dim_1, 7, 7).to(x.device)
        h2 = torch.zeros(batch_size, self.hidden_dim_2, 7, 7).to(x.device)
        
        # Sequential processing through ConvGRU
        for t in range(self.num_frames):
            h1 = self.convgru1(frame_features[t], h1)
            h2 = self.convgru2(h1, h2)
        
        # Global pooling and classification
        pooled = self.pool(h2)  # [batch, hidden_dim_2, 1, 1]
        output = self.classifier(pooled)  # [batch, num_classes]
        
        return output

# Test model instantiation
model = EmotionConvGRU(
    num_classes=Config.NUM_CLASSES,
    num_frames=Config.NUM_FRAMES,
    hidden_dim_1=Config.HIDDEN_DIM_1,
    hidden_dim_2=Config.HIDDEN_DIM_2
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(" Model instantiated successfully")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Model size: {total_params * 4 / 1e6:.2f} MB (fp32)")

 Model instantiated successfully
   Total parameters: 64,212,038
   Trainable parameters: 64,212,038
   Model size: 256.85 MB (fp32)


In [5]:
# ============================================================================
# CELL 5: FAST DATASET CLASS (NO PRE-VERIFICATION)
# ============================================================================

class GIFMultiFrameDataset(Dataset):
    """
    Fast dataset for loading GIFs with multiple frames
    NO pre-verification - handles errors on-the-fly
    """
    def __init__(self, csv_file, gif_dir, transform=None, num_frames=3, 
                 emotion_to_idx=None):
        self.df = pd.read_csv(csv_file)
        self.gif_dir = gif_dir
        self.transform = transform
        self.num_frames = num_frames
        self.emotion_to_idx = emotion_to_idx or Config.EMOTION_TO_IDX
        
        print(f" Loaded {len(self.df)} samples")
        print(f"   Emotion distribution:")
        for emotion, count in self.df['emotion_group'].value_counts().items():
            print(f"      {emotion}: {count}")
    
    def extract_frames(self, gif_path):
        """
        Extract evenly-spaced frames from GIF
        FAST version with minimal error checking
        """
        try:
            gif = Image.open(gif_path)
            
            # FAST frame counting - just try to seek
            n_frames = 0
            max_frames = 1000  # Safety limit
            while n_frames < max_frames:
                try:
                    gif.seek(n_frames)
                    n_frames += 1
                except EOFError:
                    break
            
            if n_frames == 0:
                n_frames = 1  # At least 1 frame
            
            # Calculate frame indices
            if n_frames <= self.num_frames:
                indices = list(range(self.num_frames))[:n_frames] + [n_frames-1] * (self.num_frames - n_frames)
            else:
                indices = [int(i * (n_frames - 1) / (self.num_frames - 1)) for i in range(self.num_frames)]
            
            # Extract frames QUICKLY
            frames = []
            for idx in indices:
                try:
                    gif.seek(idx)
                    frame = gif.convert('RGB')
                    if self.transform:
                        frame = self.transform(frame)
                    frames.append(frame)
                except Exception:
                    # If single frame fails, duplicate last good frame
                    if frames:
                        frames.append(frames[-1])
                    else:
                        # Create black frame
                        black = Image.new('RGB', (224, 224), (0, 0, 0))
                        if self.transform:
                            black = self.transform(black)
                        frames.append(black)
            
            return torch.stack(frames[:self.num_frames])
            
        except Exception as e:
            # Complete failure - return black frames
            black_frame = torch.zeros(3, Config.IMG_SIZE, Config.IMG_SIZE)
            return torch.stack([black_frame] * self.num_frames)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        gif_id = row['gif_id']
        gif_path = os.path.join(self.gif_dir, f"{gif_id}.gif")
        
        frames = self.extract_frames(gif_path)
        emotion = row['emotion_group']
        label = self.emotion_to_idx[emotion]
        
        return frames, label

print(" Fast Dataset class defined")

 Fast Dataset class defined


In [6]:
# ============================================================================
# CELL 6: DATA TRANSFORMS
# ============================================================================

# Training transforms (with augmentation)
train_transform = transforms.Compose([
    transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])
])

# Validation/Test transforms (no augmentation)
val_test_transform = transforms.Compose([
    transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])
])

print(" Transforms defined")
print(f"   Training: Augmentation={'enabled' if Config.USE_AUGMENTATION else 'disabled'}")

 Transforms defined
   Training: Augmentation=enabled


In [7]:
# ============================================================================
# CELL 7: CREATE DATALOADERS (SIMPLE & FAST)
# ============================================================================

print("\n Creating datasets...")

train_dataset = GIFMultiFrameDataset(
    csv_file=Config.TRAIN_CSV,
    gif_dir=Config.DATA_DIR,
    transform=train_transform if Config.USE_AUGMENTATION else val_test_transform,
    num_frames=Config.NUM_FRAMES
)

val_dataset = GIFMultiFrameDataset(
    csv_file=Config.VAL_CSV,
    gif_dir=Config.DATA_DIR,
    transform=val_test_transform,
    num_frames=Config.NUM_FRAMES
)

test_dataset = GIFMultiFrameDataset(
    csv_file=Config.TEST_CSV,
    gif_dir=Config.DATA_DIR,
    transform=val_test_transform,
    num_frames=Config.NUM_FRAMES
)

print("\n Creating dataloaders...")

# Create dataloaders - START WITH num_workers=0 (single-threaded, slower but stable)
train_loader = DataLoader(
    train_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    num_workers=0,  # Single-threaded - most stable
    pin_memory=False  # Disable for stability
)

val_loader = DataLoader(
    val_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print(" Dataloaders created")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

# Test dataloader
print("\n Testing dataloader...")
print("   Loading first batch (may take 10-30 seconds)...")

try:
    import time
    start = time.time()
    frames, labels = next(iter(train_loader))
    elapsed = time.time() - start
    
    print(f"    Success! Loaded in {elapsed:.1f} seconds")
    print(f"   Batch shape: {frames.shape}")
    print(f"   Labels shape: {labels.shape}")
    print(f"   Sample labels: {labels[:5]}")
    
except Exception as e:
    print(f"    Failed: {e}")
    import traceback
    traceback.print_exc()



 Creating datasets...
 Loaded 4145 samples
   Emotion distribution:
      positive_energetic: 1262
      negative_intense: 1023
      negative_subdued: 684
      positive_calm: 681
      surprise: 372
      contempt: 123
 Loaded 889 samples
   Emotion distribution:
      positive_energetic: 271
      negative_intense: 219
      positive_calm: 147
      negative_subdued: 146
      surprise: 79
      contempt: 27
 Loaded 889 samples
   Emotion distribution:
      positive_energetic: 271
      negative_intense: 219
      negative_subdued: 147
      positive_calm: 146
      surprise: 80
      contempt: 26

 Creating dataloaders...
 Dataloaders created
   Train batches: 260
   Val batches: 56
   Test batches: 56

 Testing dataloader...
   Loading first batch (may take 10-30 seconds)...
    Success! Loaded in 1.4 seconds
   Batch shape: torch.Size([16, 3, 3, 224, 224])
   Labels shape: torch.Size([16])
   Sample labels: tensor([4, 4, 3, 1, 4])
